# TP3 : consumer groups, offsets, rétention et compaction

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 14:05-15:00, après la démo du TP (14:00-14:05) |
| Compétences | C5 Produire, consommer, gérer les groupes et les offsets ; C6 Rétention et compaction |
| Slides de rappel | S11 (consumer groups), S12 (rééquilibrage et KIP-848), S29 (commits et `auto.offset.reset`), S31 (rétention et compaction) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule, avec trois terminaux |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- La version Markdown de ce TP utilise trois terminaux pour faire tourner plusieurs consommateurs. Ici, chaque consommateur est une **instance** qui tourne en arrière-plan dans le notebook (classe `Instance`, étape 7.3).
- Les fonctions `offsets_du_topic()` et `offsets_de_fin()` reprennent le TP2 : elles sont fournies dans la cellule suivante.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import csv
import re
import threading
import time

from confluent_kafka import (OFFSET_BEGINNING, Consumer, ConsumerGroupState, ConsumerGroupTopicPartitions,
                             ConsumerGroupType, KafkaException, Producer, TopicPartition)
from confluent_kafka.admin import NewTopic, OffsetSpec


def offsets_du_topic(topic, spec):
    """{partition: offset} pour une specification : OffsetSpec.earliest(), latest() ou for_timestamp(ms)."""
    admin = AdminClient(CONF)
    nb = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    futurs = admin.list_offsets({TopicPartition(topic, p): spec for p in range(nb)})
    return {tp.partition: f.result().offset for tp, f in sorted(futurs.items(), key=lambda x: x[0].partition)}


def offsets_de_fin(topic):
    return offsets_du_topic(topic, OffsetSpec.latest())


print("Bootstrap INTERNAL :", BS)
print("orders :", offsets_de_fin("orders"))

## 1. Objectifs

À la fin du TP, vous savez :

- consommer avec un consumer group et calculer son lag ;
- observer la répartition des partitions entre consommateurs et le rééquilibrage ;
- reconnaître un groupe utilisant le nouveau protocole (KIP-848) ;
- repositionner les offsets d'un groupe en sécurité : simulation, application, export ;
- faire supprimer des données par la rétention en temps ;
- configurer un topic compacté et supprimer une clé avec un tombstone.

## 2. Durée

55 minutes, après 5 minutes de démonstration.

| Étape | Durée |
|---|---|
| 7.1 Consommer avec un groupe | 5 min |
| 7.2 Créer et résorber du lag | 5 min |
| 7.3 Deux consommateurs dans un groupe | 7 min |
| 7.4 Départ d'un consommateur | 4 min |
| 7.5 Le nouveau protocole de groupe | 5 min |
| 7.6 Réinitialiser les offsets | 7 min |
| 7.7 Défi : retraiter la dernière heure | 4 min |
| 7.8 Rétention en temps | 7 min |
| 7.9 Compaction | 6 min |
| 7.10 Tombstone | 5 min |

## 3. Prérequis

- TP2 terminé : topic `orders` avec 6 partitions et 60 messages (voir la cellule précédente). Sinon, demandez au formateur le bloc de rattrapage du TP2.
- Jeu de données `lab/data/product-prices.txt` : 20 prix pour 4 produits `P-100` à `P-400`.
- Notions : slides S11, S29, S31.

## 4. Architecture utilisée

```
                       orders (6 partitions)
          p0    p1    p2    p3    p4    p5
           \     |     /     \     |     /
   groupe analytics : instance a (p0, p1, p2) + instance b (p3, p4, p5)
   groupe billing   : un consommateur, lancé puis arrêté (offsets conservés)

   Offsets validés : topic interne __consumer_offsets (50 partitions, compacté)
   Coordinateur du groupe : le leader de la partition de __consumer_offsets
   qui correspond au nom du groupe.
```

| Topic | Partitions | Configuration | Usage |
|---|---|---|---|
| `orders` | 6 | défaut | Groupes et offsets |
| `audit-short` | 1 | `retention.ms=60000`, `segment.ms=10000` | Rétention en temps |
| `product-prices` | 1 | `cleanup.policy=compact`, segments de 10 s | Compaction |

## 5. Contexte métier

Chez NovaShop, deux services lisent les commandes :

- **Facturation** (groupe `billing`) doit traiter chaque commande une fois, et pouvoir rejouer une période après un incident ;
- **Analytique** (groupe `analytics`) doit monter en charge en ajoutant des instances.

Le service Audit veut un journal court, et le service Catalogue un référentiel de prix qui ne garde que le dernier prix de chaque produit. L'état des groupes s'affiche en fin de notebook.

## 6. Énoncé

1. Faites consommer `orders` par la facturation, puis montrez le lag créé par de nouvelles commandes.
2. Faites travailler deux instances de l'analytique en parallèle, et observez ce qui se passe quand l'une s'arrête.
3. Repositionnez la facturation pour rejouer des commandes, sans jamais appliquer un changement que vous n'avez pas simulé.
4. Prouvez que la rétention supprime les vieux messages d'`audit-short` et que la compaction ne garde que le dernier prix de chaque produit.

## 7. Étapes

### 7.1 Consommer avec un groupe

Un consommateur qui appartient à un groupe valide (« commit ») ses offsets dans le topic interne `__consumer_offsets`. À son redémarrage, il reprend à ces offsets. `auto.offset.reset` ne sert que pour une partition où le groupe n'a encore rien validé : `earliest` lit alors depuis le début, `latest` (valeur par défaut) seulement les nouveaux messages.

**Exemple en ligne de commande** : les groupes existants.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-consumer.sh --bootstrap-server $BS --topic orders --group billing \
  --from-beginning --timeout-ms 10000 | wc -l
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing
```

In [ ]:
# TODO 7.1 : consommer tout orders avec le groupe billing, puis calculer son lag
# 1. Completez consommer(groupe, topic) : elle lit jusqu'a ce qu'aucun message n'arrive pendant 5 s,
#    ferme le consommateur et renvoie la liste des messages lus.
#    consommateur.consume(100, timeout=1) renvoie une liste de messages (vide si rien n'arrive en 1 s).
def consommer(groupe, topic, **config):
    # earliest : un groupe sans offset valide lit depuis le debut.
    # Les offsets sont valides automatiquement (enable.auto.commit), et une derniere fois par close().
    consommateur = Consumer({"bootstrap.servers": BS, "group.id": groupe, "auto.offset.reset": "earliest",
                             "log_level": 4, **config})
    consommateur.subscribe([topic])
    messages = []
    # a completer
    consommateur.close()
    return messages


# 2. Ecrivez offsets_valides(groupe, topic) : {partition: offset valide}.
#    admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(groupe)]) renvoie {groupe: futur} ;
#    le resultat a un attribut topic_partitions : liste de TopicPartition (topic, partition, offset).
#    Une partition absente de la liste, ou d'offset negatif, n'a rien de valide : comptez 0.
def offsets_valides(groupe, topic):
    return {}


# 3. Ecrivez lag(groupe, topic) : {partition: offset de fin - offset valide}, avec offsets_de_fin().
def lag(groupe, topic):
    return {}


lus = consommer("billing", "orders")
print(len(lus), "messages lus")
print("lag :", lag("billing", "orders"))

In [ ]:
# Verification 7.1
admin = AdminClient(CONF)
assert len(lus) == 60, "A revoir : 60 messages lus (%d)" % len(lus)
print("OK  60 messages lus (%d)" % len(lus))
assert lag("billing", "orders") == {p: 0 for p in range(6)}, "A revoir : lag nul sur les 6 partitions"
print("OK  lag nul sur les 6 partitions")
etat = admin.describe_consumer_groups(["billing"])["billing"].result().state
assert etat == ConsumerGroupState.EMPTY, "A revoir : groupe billing sans membre actif : %s" % etat.name
print("OK  groupe billing sans membre actif : %s" % etat.name)

**Résultat attendu** : 60 messages lus, en une dizaine de secondes. Le lag est nul partout et le groupe est `EMPTY` : il n'a plus de membre, mais ses offsets sont conservés.

### 7.2 Créer et résorber du lag

Le lag d'une partition est l'écart entre son offset de fin et l'offset validé du groupe : le nombre de messages qui restent à traiter.

Équivalent en ligne de commande de l'envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
for i in $(seq 1 10); do
  echo "C008|{\"order_id\":\"O-30$i\",\"customer\":\"C008\",\"amount\":10.0,\"city\":\"Lyon\"}"
done | kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

**Exécutez la cellule `TODO` une seule fois** : chaque exécution envoie 10 commandes.

In [ ]:
# TODO 7.2 (a) : envoyer 10 commandes du nouveau client C008, puis mesurer le lag de billing
# Le producteur a la meme configuration qu'au TP2 : les cles sont rangees comme avec les outils Java.
producteur = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4})
# Valeur de la commande i (i de 1 a 10) :
#   '{"order_id":"O-30%d","customer":"C008","amount":10.0,"city":"Lyon"}' % i
# Envoyez les 10 commandes avec la cle "C008", attendez les confirmations (flush),
# puis rangez lag("billing", "orders") dans lag_apres.
lag_apres = {}

print("lag :", lag_apres)

In [ ]:
# Verification 7.2 (a)
assert lag_apres == {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 10}, "A revoir : lag de 10 sur la partition 5 seulement : %s" % lag_apres
print("OK  lag de 10 sur la partition 5 seulement : %s" % lag_apres)

**Exemple en ligne de commande** : la même mesure avec l'outil Java.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

In [ ]:
# TODO 7.2 (b) : relancer la facturation et compter les messages lus
# Relancez consommer() avec le groupe billing et rangez la liste des messages dans relus.
relus = []

print(len(relus), "messages relus :", sorted({m.key().decode() for m in relus}))

In [ ]:
# Verification 7.2 (b)
assert len(relus) == 10 and {m.key().decode() for m in relus} == {"C008"}, "A revoir : 10 commandes de C008 relues"
print("OK  10 commandes de C008 relues")
assert sum(lag("billing", "orders").values()) == 0, "A revoir : lag resorbe"
print("OK  lag resorbe")

**Résultat attendu** : toutes les commandes de `C008` vont dans la partition 5 ; le lag vaut 10 sur cette partition, 0 ailleurs. La relance lit exactement les 10 nouvelles commandes : le groupe reprend à ses offsets validés.

### 7.3 Deux consommateurs dans un groupe

Les consommateurs d'un même groupe se partagent les partitions : chaque partition est lue par un seul membre. La cellule suivante définit `Instance`, une instance d'un service consommateur qui tourne en arrière-plan dans le notebook, comme un terminal de la version Markdown :

- `Instance("analytics")` démarre un consommateur du groupe `analytics`, abonné à `orders` ;
- `instance.affectation` donne ses partitions, mises à jour en continu ;
- `instance.arreter()` le ferme proprement : il quitte le groupe ;
- `Instance.arreter_tout()` arrête toutes les instances lancées.

In [ ]:
class Instance:
    """Une instance d'un service consommateur : un Consumer qui tourne dans un fil d'execution."""

    lancees = []

    def __init__(self, groupe, topic="orders", **config):
        self.groupe = groupe
        self.affectation = []
        self._arret = threading.Event()
        self.consommateur = Consumer({"bootstrap.servers": BS, "group.id": groupe,
                                      "auto.offset.reset": "earliest", "log_level": 4, **config})
        self.consommateur.subscribe([topic])
        self._fil = threading.Thread(target=self._boucle, daemon=True)
        self._fil.start()
        Instance.lancees.append(self)

    def _boucle(self):
        while not self._arret.is_set():
            # consume() fait aussi vivre l'appartenance au groupe (reequilibrages, validations)
            self.consommateur.consume(100, timeout=0.5)
            self.affectation = sorted(tp.partition for tp in self.consommateur.assignment())
        # Depart propre : le coordinateur est prevenu et reequilibre aussitot
        self.consommateur.close()

    def arreter(self):
        self._arret.set()
        self._fil.join(15)

    @classmethod
    def arreter_tout(cls):
        for instance in cls.lancees:
            instance.arreter()
        cls.lancees = []

    def __repr__(self):
        return "Instance(%s, partitions %s)" % (self.groupe, self.affectation)

In [ ]:
# TODO 7.3 : deux instances du service analytics
#   1. Lancez deux instances du groupe analytics, dans les variables a et b.
#   2. Attendez que chacune ait des partitions : attendre(lambda: a.affectation and b.affectation, timeout=60).
#   3. Decrivez le groupe : admin.describe_consumer_groups(["analytics"]) renvoie {groupe: futur}.
#      Le resultat a les attributs state, type, partition_assignor et members ; chaque membre a
#      member_id, client_id et assignment.topic_partitions. Rangez-le dans groupe_analytics.
#   4. Demandez le retour au debut des offsets d'analytics pendant qu'il est actif :
#        admin.alter_consumer_group_offsets([ConsumerGroupTopicPartitions("analytics",
#            [TopicPartition("orders", p, 0) for p in range(6)])])
#      renvoie {groupe: futur}. Rangez dans erreurs_reset les codes d'erreur des partitions
#      (tp.error.name() pour chaque tp de resultat.topic_partitions dont tp.error n'est pas None).
a = b = None
groupe_analytics = None
erreurs_reset = []

print(a, b)

In [ ]:
# Verification 7.3
assert (a is not None and b is not None and sorted(a.affectation + b.affectation) == list(range(6))
         and len(a.affectation) == 3), "A revoir : 3 partitions par instance : %s et %s" % (a, b)
print("OK  3 partitions par instance : %s et %s" % (a, b))
assert (groupe_analytics is not None and groupe_analytics.state == ConsumerGroupState.STABLE
         and len(groupe_analytics.members) == 2), "A revoir : groupe STABLE avec 2 membres"
print("OK  groupe STABLE avec 2 membres")
assert groupe_analytics.partition_assignor == "range", "A revoir : strategie range"
print("OK  strategie range")
assert len(erreurs_reset) == 6 and set(erreurs_reset) == {"UNKNOWN_MEMBER_ID"}, "A revoir : repositionnement refuse sur les 6 partitions : %s" % sorted(set(erreurs_reset))
print("OK  repositionnement refuse sur les 6 partitions : %s" % sorted(set(erreurs_reset)))

**Vue en ligne de commande**, pendant que les deux instances tournent. Le repositionnement est demandé en simulation (`--dry-run`) : l'outil le refuse aussi.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group analytics --topic orders --reset-offsets --to-earliest --dry-run

**Résultat attendu** : chaque instance reçoit 3 partitions, `[0, 1, 2]` et `[3, 4, 5]`. État `STABLE`, 2 membres, stratégie `range`, client `rdkafka`. Le repositionnement est refusé partition par partition (`UNKNOWN_MEMBER_ID`) ; l'outil Java refuse avant d'envoyer quoi que ce soit : `Assignments can only be reset if the group 'analytics' is inactive, but the current state is Stable.`

### 7.4 Départ d'un consommateur

Quand un membre quitte le groupe, le coordinateur rééquilibre : il redistribue ses partitions aux membres restants.

In [ ]:
# TODO 7.4 : arreter l'instance b et suivre le groupe
#   1. Notez debut = time.time(), puis arretez b.
#   2. Toutes les 0,5 s, pendant au plus 20 s, decrivez le groupe analytics et ajoutez a etats
#      le couple (secondes depuis debut arrondies au dixieme, groupe.state.name).
#      Arretez-vous quand l'etat est STABLE et que a.affectation compte 6 partitions.
#   3. Arretez a, puis rangez dans etat_final le nom de l'etat du groupe.
etats = []
etat_final = None

print(etats)
print("a :", a, "- etat final :", etat_final)

In [ ]:
# Verification 7.4
assert etats and etats[-1][1] == "STABLE", "A revoir : groupe de nouveau STABLE apres le depart de b"
print("OK  groupe de nouveau STABLE apres le depart de b")
assert a is not None and a.affectation == list(range(6)), "A revoir : a a recu les 6 partitions"
print("OK  a a recu les 6 partitions")
assert etat_final == "EMPTY", "A revoir : groupe EMPTY apres l'arret de a"
print("OK  groupe EMPTY apres l'arret de a")

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state

**Résultat attendu** : en général `PREPARING_REBALANCING` pendant une à deux secondes, puis `STABLE` ; l'instance `a` reçoit les 6 partitions. Après son arrêt, le groupe est `EMPTY`, sans membre.

### 7.5 Le nouveau protocole de groupe

Avec le nouveau protocole (KIP-848, `group.protocol=consumer`), le coordinateur calcule lui-même l'affectation cible, et chaque membre converge vers elle sans arrêt global du groupe. Le protocole classique reste la valeur par défaut des clients en Kafka 4.3.

In [ ]:
# TODO 7.5 (a) : le groupe analytics-v2 avec le nouveau protocole
#   1. Lancez deux instances du groupe analytics-v2 avec l'option group.protocol=consumer, dans c et d :
#        Instance("analytics-v2", **{"group.protocol": "consumer"})
#   2. Attendez que les 6 partitions soient reparties entre les deux (attendre, timeout=60).
#   3. Decrivez le groupe dans groupe_v2. Chaque membre a aussi un attribut target_assignment.
#   4. admin.list_consumer_groups().result().valid est la liste des groupes : chacun a group_id
#      et type (type.name vaut CLASSIC ou CONSUMER). Rangez {group_id: type.name} dans types.
c = d = None
groupe_v2 = None
types = {}

print(c, d)
print(types)

In [ ]:
# Verification 7.5 (a)
assert groupe_v2 is not None and groupe_v2.type == ConsumerGroupType.CONSUMER, "A revoir : analytics-v2 de type CONSUMER"
print("OK  analytics-v2 de type CONSUMER")
assert groupe_v2.partition_assignor == "uniform", "A revoir : affectation calculee par le serveur : uniform"
print("OK  affectation calculee par le serveur : uniform")
assert len(groupe_v2.members) == 2, "A revoir : 2 membres"
print("OK  2 membres")
assert (types.get("analytics") == "CLASSIC" and types.get("billing") == "CLASSIC"
         and types.get("analytics-v2") == "CONSUMER"), "A revoir : types des groupes : %s" % types
print("OK  types des groupes : %s" % types)

**Vue en ligne de commande**, pendant que les deux instances tournent :

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics-v2 --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list --type

In [ ]:
# TODO 7.5 (b) : arreter les deux instances, puis rangez dans etat_v2 le couple
# (nom de l'etat, partition_assignor) du groupe analytics-v2.
etat_v2 = (None, None)

print(etat_v2)

In [ ]:
# Verification 7.5 (b)
assert etat_v2 == ("EMPTY", "uniform"), "A revoir : analytics-v2 vide, assignateur uniform : %s" % (etat_v2,)
print("OK  analytics-v2 vide, assignateur uniform : %s" % (etat_v2,))

**Résultat attendu** : `type` `CONSUMER`, assignateur `uniform`, 2 membres dont l'affectation est égale à la cible. Les colonnes `CURRENT-EPOCH` et `TARGET-EPOCH` de l'outil Java sont renseignées. `--list --type` affiche `Consumer` pour `analytics-v2` et `Classic` pour les autres groupes.

### 7.6 Réinitialiser les offsets

Le groupe `billing` est inactif. On peut repositionner ses offsets, en simulant toujours avant d'appliquer.

**Exemple en ligne de commande** : simulation d'un retour au début. `--dry-run` affiche les nouveaux offsets sans rien modifier.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --dry-run

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-latest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --shift-by -5 --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders \
  --reset-offsets --to-current --export --dry-run > /tmp/billing-offsets.csv
```

In [ ]:
# TODO 7.6 : repositionner billing, toujours en simulant d'abord
# 1. Ecrivez repositionner(groupe, topic, nouveaux, appliquer=False). nouveaux est un dictionnaire
#    {partition: nouvel offset}. La fonction :
#      - affiche le tableau partition / offset valide / nouvel offset (c'est la simulation) ;
#      - si appliquer vaut True, applique avec admin.alter_consumer_group_offsets(...) (voir 7.3)
#        et leve RuntimeError si une partition porte une erreur.
def repositionner(groupe, topic, nouveaux, appliquer=False):
    pass


# 2. Retour au debut : simulez, puis appliquez offsets_du_topic("orders", OffsetSpec.earliest()).
#    Rangez le lag total (somme des valeurs de lag()) dans lag_debut.
# 3. Placez billing a la fin, puis reculez de 5 messages par partition, sans descendre sous
#    le premier offset disponible. Rangez le lag total dans lag_recul.
# 4. Exportez les offsets valides dans /tmp/billing-offsets.csv : une ligne "topic,partition,offset"
#    par partition, comme l'export de l'outil Java.
lag_debut = lag_recul = None

print("lag_debut :", lag_debut, "- lag_recul :", lag_recul)

In [ ]:
# Verification 7.6
assert lag_debut == 70, "A revoir : apres retour au debut, lag egal au nombre de messages : %s" % lag_debut
print("OK  apres retour au debut, lag egal au nombre de messages : %s" % lag_debut)
assert lag_recul == 20, "A revoir : apres recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_recul
print("OK  apres recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_recul)
try:
    lignes = open("/tmp/billing-offsets.csv").read().split()
except OSError:
    lignes = []
assert len(lignes) == 6 and all(l.startswith("orders,") for l in lignes), "A revoir : export CSV de 6 lignes"
print("OK  export CSV de 6 lignes")

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

**Résultat attendu** : après le retour au début, un lag total de 70, tous les messages du topic. Après le recul de 5, un lag de 20 : 5 sur chacune des partitions 0, 1, 2 et 5 ; les partitions vides 3 et 4 restent à 0. Le fichier contient une ligne par partition, de `orders,0,7` à `orders,5,16`.

### 7.7 Défi : retraiter la dernière heure

La facturation a mal traité toutes les commandes reçues depuis une heure. Repositionnez `billing` pour les retraiter, en simulant d'abord.

**Exemple en ligne de commande** : les options de l'outil Java qui parlent de temps.

In [ ]:
!docker exec kafka-tools bash -c "kafka-consumer-groups.sh --help 2>&1 | grep -A2 -E '^--(by-duration|to-datetime)'"

In [ ]:
# TODO 7.7 (defi) : repositionner billing sur les commandes recues depuis une heure
# OffsetSpec.for_timestamp(ms) donne, pour chaque partition, le premier offset dont l'horodatage
# est posterieur ou egal a ms (millisecondes depuis 1970), ou -1 si aucun message n'est aussi recent.
# time.time() donne l'heure courante en secondes.
# Construisez nouveaux_1h {partition: offset}, simulez avec repositionner(), puis appliquez.
nouveaux_1h = {}

print(nouveaux_1h)

In [ ]:
# Verification 7.7 : comparaison avec la simulation de l'outil Java
sortie = !docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --by-duration PT1H --dry-run 2>&1
sortie = sortie.n
attendu_java = {int(p): int(o) for p, o in re.findall(r"^billing\s+orders\s+(\d+)\s+(\d+)", sortie, re.M)}
print("outil Java :", attendu_java)
assert nouveaux_1h and nouveaux_1h == attendu_java, "A revoir : memes offsets que --by-duration PT1H"
print("OK  memes offsets que --by-duration PT1H")
assert offsets_valides("billing", "orders") == nouveaux_1h, "A revoir : repositionnement applique"
print("OK  repositionnement applique")

**Résultat attendu** : si le TP2 date de moins d'une heure, l'offset 0 partout, comme `--to-earliest`. Si le déjeuner les sépare, seules les commandes de `C008` (étape 7.2) ont moins d'une heure : la partition 5 revient à l'offset 11, les autres partitions sont placées à la fin.

### 7.8 Rétention en temps

La rétention supprime des **segments** entiers, jamais des messages isolés. Un segment est fermé quand il atteint `segment.bytes` ou `segment.ms` : le message suivant ouvre un nouveau segment. Un segment est supprimé quand son message le plus récent dépasse `retention.ms`, y compris le segment actif (celui où le broker écrit) : le broker en ouvre alors un nouveau, vide.

**Exemple en ligne de commande** : la fréquence de vérification de la rétention, réglage de lab.

In [ ]:
!grep -B1 log.retention.check.interval.ms config/kafka/kafka-1/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic audit-short --partitions 1 --replication-factor 3 \
  --config retention.ms=60000 --config segment.ms=10000
seq 1 100 | sed 's/^/audit-/' | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
sleep 35; echo audit-101 | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
kafka-get-offsets.sh --bootstrap-server $BS --topic audit-short --time earliest
```

Pourquoi 35 s : plus que `segment.ms` (10 s), pour que `audit-101` ouvre un nouveau segment ; et plus que la période de vérification de la rétention (30 s dans le lab), pour qu'une vérification passe entre l'expiration du premier segment et celle du second.

In [ ]:
# TODO 7.8 : faire supprimer des messages par la retention
#   1. Creez audit-short : 1 partition, RF 3, avec retention.ms=60000 et segment.ms=10000
#      (NewTopic(..., config={"retention.ms": "60000", ...})).
#   2. Envoyez "audit-1" a "audit-100" (producteur), attendez 35 s (time.sleep), puis envoyez "audit-101".
#   3. Attendez que le premier offset disponible de la partition 0,
#      offsets_du_topic("audit-short", OffsetSpec.earliest())[0], atteigne au moins 100 :
#      attendre(..., timeout=180, pas=5). Rangez la duree d'attente en secondes dans duree_retention.
duree_retention = None

print("duree_retention :", duree_retention)

In [ ]:
# Verification 7.8
assert duree_retention is not None, "A revoir : suppression du premier segment attendue (%s s)" % duree_retention
print("OK  suppression du premier segment attendue (%s s)" % duree_retention)
premier_offset = offsets_du_topic("audit-short", OffsetSpec.earliest())[0]
assert premier_offset in (100, 101), "A revoir : premier offset disponible : %d (100, puis 101 quand le segment actif expire a son tour)" % premier_offset
print("OK  premier offset disponible : %d (100, puis 101 quand le segment actif expire a son tour)" % premier_offset)
assert offsets_de_fin("audit-short") == {0: 101}, "A revoir : 101 messages ecrits"
print("OK  101 messages ecrits")

In [ ]:
!docker compose exec -T kafka-1 ls -l /var/lib/kafka/data/audit-short-0

**Résultat attendu** :

- d'abord `{0: 0}` et `{0: 100}` ;
- entre 1 min et 1 min 40 après le 100e message, le premier offset passe à 100. Dans le répertoire de la partition, l'ancien segment est renommé `00000000000000000000.log.deleted`, puis effacé une minute plus tard ; le segment actif est `00000000000000000100.log` ;
- 30 à 60 s plus tard, le premier offset passe à 101 : le segment actif, qui ne contient que `audit-101`, a expiré à son tour. La vérification accepte donc 100 ou 101, selon le moment où vous l'exécutez.

### 7.9 Compaction

Un topic compacté (`cleanup.policy=compact`) garde au moins le dernier message de chaque clé. Le nettoyeur (« log cleaner ») ne travaille que sur les segments fermés.

La fonction fournie ci-dessous lit tous les messages encore présents dans une partition. Avec un topic compacté, les offsets ne se suivent plus : elle lit jusqu'au dernier offset du topic au lieu de compter les messages.

In [ ]:
def lire_tout(topic, partition=0):
    """Messages encore presents dans une partition : liste de (offset, cle, valeur)."""
    fin = offsets_de_fin(topic)[partition]
    consommateur = Consumer({"bootstrap.servers": BS, "group.id": "tp3-lecture",
                             "enable.auto.commit": False, "log_level": 4})
    consommateur.assign([TopicPartition(topic, partition, OFFSET_BEGINNING)])
    messages = []
    while fin and (not messages or messages[-1][0] < fin - 1):
        lot = consommateur.consume(100, timeout=5)
        if not lot:
            break
        for m in lot:
            if m.error() is None:
                cle = m.key().decode() if m.key() is not None else None
                valeur = m.value().decode() if m.value() is not None else None
                messages.append((m.offset(), cle, valeur))
    consommateur.close()
    return messages

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic product-prices --partitions 1 --replication-factor 3 \
  --config cleanup.policy=compact --config segment.ms=10000 \
  --config min.cleanable.dirty.ratio=0.01 --config delete.retention.ms=1000
kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' < /opt/lab/data/product-prices.txt
sleep 12
echo 'P-500|9.99' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

In [ ]:
# TODO 7.9 : un referentiel de prix compacte
#   1. Creez product-prices : 1 partition, RF 3, avec cleanup.policy=compact, segment.ms=10000,
#      min.cleanable.dirty.ratio=0.01 et delete.retention.ms=1000.
#   2. Envoyez les 20 lignes de "data/product-prices.txt" (format "produit|prix",
#      le produit est la cle), attendez 12 s, puis envoyez la cle "P-500" avec la valeur "9.99".
#   3. Attendez que lire_tout("product-prices") ne renvoie plus que 5 messages
#      (attendre, timeout=180, pas=5), et rangez la liste finale dans prix.
prix = []

tableau(prix, ["offset", "cle", "valeur"])

In [ ]:
# Verification 7.9
assert len(prix) == 5 and sorted(m[1] for m in prix) == ["P-100", "P-200", "P-300", "P-400", "P-500"], "A revoir : un seul prix par produit"
print("OK  un seul prix par produit")
assert prix[0][0] > 0, "A revoir : les offsets conserves ne se suivent pas : premier offset %s" % (prix[0][0] if prix else None)
print("OK  les offsets conserves ne se suivent pas : premier offset %s" % (prix[0][0] if prix else None))

**Résultat attendu** : 21 messages juste après l'envoi de `P-500`, puis 5 en moins d'une minute : le dernier prix de `P-100`, `P-200`, `P-300`, `P-400` et `P-500`, aux offsets 16 à 20.

### 7.10 Tombstone

Pour supprimer une clé d'un topic compacté, on envoie un **tombstone** : un message avec cette clé et une valeur nulle (`value=None` en Python). Le nettoyeur supprime d'abord les anciennes valeurs, puis, plus tard, le tombstone lui-même.

Équivalent en ligne de commande de l'envoi du tombstone (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
echo 'P-500|NULL' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' --reader-property null.marker=NULL
```

In [ ]:
# TODO 7.10 : supprimer P-500 avec un tombstone, et observer le nettoyage
#   1. Envoyez la cle "P-500" avec la valeur None.
#   2. Relevez aussitot les valeurs lisibles de P-500 :
#        [v for (o, k, v) in lire_tout("product-prices") if k == "P-500"]
#      et ajoutez a etapes le couple (0, valeurs).
#   3. Puis toutes les 15 s, pendant au plus 3 minutes : envoyez un message "P-90i" (i = 1, 2...)
#      pour faire travailler le nettoyeur, relevez les valeurs de P-500 et ajoutez
#      (secondes ecoulees, valeurs) a etapes. Arretez-vous quand plus aucune valeur n'est lisible.
etapes = []

for t, valeurs in etapes:
    print("t+%3d s : %s" % (t, valeurs))

In [ ]:
# Verification 7.10
assert etapes and etapes[0][1] == ["9.99", None], "A revoir : au debut, l'ancienne valeur et le tombstone coexistent"
print("OK  au debut, l'ancienne valeur et le tombstone coexistent")
assert etapes[-1][1] == [], "A revoir : a la fin, plus aucune trace de P-500"
print("OK  a la fin, plus aucune trace de P-500")

**Résultat attendu** :

- d'abord, l'ancienne valeur `9.99` et le tombstone `None` coexistent ;
- ensuite, l'ancienne valeur disparaît, le tombstone reste ;
- enfin, environ une minute après l'envoi, plus aucune valeur pour `P-500`.

## État des groupes

La cellule suivante résume l'état des groupes de consommateurs à la fin du TP.

In [ ]:
admin = AdminClient(CONF)
def etat_groupe(nom):
    admin = AdminClient(CONF)
    g = admin.describe_consumer_groups([nom])[nom].result()
    retard = lag(nom, "orders")
    return (nom, g.type.name, g.state.name, len(g.members), sum(retard.values()) if retard else "?")


existants = {g.group_id for g in admin.list_consumer_groups().result().valid}
lignes = []
for nom in ("billing", "analytics", "analytics-v2"):
    if nom not in existants:
        lignes.append((nom, "absent", "", "", ""))
        continue
    lignes.append(etat_groupe(nom))
tableau(lignes, ["Groupe", "Type", "Etat", "Membres", "Lag sur orders"])

**Attendu** : les trois groupes `EMPTY`, sans membre. `billing` est `CLASSIC`, avec le lag laissé par l'étape 7.7 ; `analytics` est `CLASSIC` et `analytics-v2` est `CONSUMER`, sans lag. Ne supprimez pas ces groupes : le TP6 les réutilise.

## 11. Questions de réflexion

1. Pourquoi la relance de la facturation en 7.2 lit-elle 10 messages et non 70, alors que `auto.offset.reset` vaut `earliest` ?
2. Que se passerait-il avec 8 instances dans le groupe `analytics` ?
3. Pourquoi Kafka refuse-t-il de repositionner les offsets d'un groupe actif ? Comment procéder en production ?
4. Qu'apporte le nouveau protocole de groupe (KIP-848) lors d'un rééquilibrage ?
5. Pourquoi le message `audit-101` a-t-il survécu aux 100 premiers, alors que la rétention est la même pour tous ?
6. Un consommateur qui lit `product-prices` depuis le début voit-il toujours exactement un message par produit ?
7. Pourquoi un tombstone n'est-il pas supprimé immédiatement ?

## 12. Erreurs fréquentes

| Symptôme | Cause | Correction |
|---|---|---|
| Plus de 2 membres dans `analytics` | Cellule de l'étape 7.3 exécutée deux fois : de nouvelles instances s'ajoutent | `Instance.arreter_tout()`, puis reprendre l'étape |
| Un membre reste 45 s après un redémarrage du noyau | Les instances ont été tuées sans quitter le groupe | Attendre `session.timeout.ms` (45 s), ou relancer et arrêter proprement |
| Repositionnement « réussi » mais rien ne change | Erreurs par partition non lues (`tp.error`), ou simulation sans `appliquer=True` | Lire `topic_partitions` du résultat ; appliquer après la simulation |
| `RuntimeError: partitions refusees ... UNKNOWN_MEMBER_ID` | Une instance du groupe tourne encore | `Instance.arreter_tout()`, vérifier l'état `EMPTY` |
| `consommer()` ne lit rien | Groupe qui a déjà tout lu : il reprend à ses offsets validés | Normal ; repositionner le groupe pour relire |
| `KafkaException ... NOT_COORDINATOR` sur une opération de groupe, ou noyau qui redémarre seul | Un `AdminClient` gardé dans une variable connaît encore l'ancien coordinateur du groupe, qui a changé de broker : le contrôleur rééquilibre les leaders 5 minutes après le redémarrage d'un nœud | Appeler `admin` pour chaque opération : elle crée un client neuf |
| Le premier offset d'`audit-short` reste à 0 | Rien n'expire avant 60 s, puis la vérification de la rétention ne passe que toutes les 30 s | Attendre : jusqu'à 1 min 40 après le 100e message |
| Le premier offset passe directement de 0 à 101 | Les deux segments ont expiré avant la même vérification : 101e message parti moins de 30 s après les autres, ou dans le même segment (moins de 10 s) | Le constat reste juste (suppression par segment) ; pour voir l'étape à 100, recommencer sur un nouveau topic avec 35 s d'écart |
| La compaction ne réduit rien | Toutes les données sont dans le segment actif | Envoyer un message après `segment.ms` pour fermer le segment |
| `FencedInstanceIdException` sur un consommateur statique | Un autre consommateur avec le même `group.instance.id` s'est enregistré : le plus ancien est rejeté | Un identifiant distinct par instance |

## 13. Solution formateur

Voir `corrige.ipynb` (et `corrige.md` pour la version en ligne de commande).

## 14. Variante avancée

1. Restaurez les offsets de `billing` à partir du fichier `/tmp/billing-offsets.csv` exporté en 7.6, après avoir placé le groupe à la fin du topic.
2. Trouvez la partition de `__consumer_offsets` qui stocke les offsets de `billing`, et vérifiez que son leader est le coordinateur du groupe (`describe_consumer_groups(...).coordinator`). La partition est `abs(hashCode("billing")) % 50`, où `hashCode` est celui des chaînes Java : `h = 31 × h + code du caractère`, calculé sur 32 bits signés.
3. Lancez trois instances dans `analytics`, puis tuez brutalement un consommateur au lieu de l'arrêter proprement. Combien de temps le groupe met-il à réaffecter ses partitions ? Quel paramètre fixe ce délai ? Indice : un consommateur console lancé dans `tools` peut être tué avec `kill -9`.
4. **Membres statiques.** Lancez deux consommateurs console du groupe `statique` sur `orders` depuis `tools`, avec `--consumer-property group.instance.id=inst-1` (puis `inst-2`) et `--consumer-property partition.assignment.strategy=org.apache.kafka.clients.consumer.CooperativeStickyAssignor`. Relevez l'état et les partitions de chaque membre, arrêtez `inst-2` (`Ctrl+C`) et regardez le groupe 5 secondes plus tard, relancez-le avec le même identifiant, puis arrêtez-le et attendez plus d'une minute. Que devient le groupe à chaque fois ? Quel paramètre fixe le délai ?